In [3]:
# 1) Pick a private key
import os
from ecpy.curves import Curve, Point
from ecpy.keys import ECPublicKey, ECPrivateKey
from sha3 import keccak_256
import numpy as np
from libnum import has_sqrtmod_prime_power, sqrtmod_prime_power

cv = Curve.get_curve('secp256k1')
order = cv.order
field = cv.field
print("Curve order:", order)

rng = np.random.default_rng()
def random_uint256():
    k = int.from_bytes(os.urandom(32), 'big')
    # Reduce modulo n and ensure k is in [1, n-1]
    k = k % order
    while k == 0:  # Handle the extremely rare case of k = 0
        k = int.from_bytes(os.urandom(32), 'big') % order
    return k


p = random_uint256()
print("Private key:", p)


Curve order: 115792089237316195423570985008687907852837564279074904382605163141518161494337
Private key: 72329000536779791609889445150063805330999049088740615955324526097706168154835


In [4]:
# 2) Generate the public key using that private key (not the eth address, the public key)

cv   = Curve.get_curve('secp256k1')
G = cv.generator
P = p * G

def calc_eth_addr(P):
    concat_x_y = P.x.to_bytes(32, byteorder='big') + P.y.to_bytes(32, byteorder='big')
    eth_addr = '0x' + keccak_256(concat_x_y).digest()[-20:].hex()
    return eth_addr

print("Public Key:", P)
print("ETH address:", calc_eth_addr(P))

Public Key: (0xa0fe012cafe7b9eb78c6589d9abc1d6feebf53b8f63b8f5dc65f31b66860fbf1 , 0xc47296ce065b9f7367dc729193cec3bf66f22fb2249b21c2ba19a4b06009d5b2)
ETH address: 0x9ebd5ccf93049c6acc236dca6664796bd3acaa5e


In [5]:
# 3) Pick message m and hash it to produce h (h can be though of as a 256 bit number)
message = "This is my first ECDSA algo.".encode('utf-8')
h = int(keccak_256(message).digest().hex(), 16) % order
print("Message:",message)
print("Message Hash:",h)

Message: b'This is my first ECDSA algo.'
Message Hash: 38294091763788184178067574739767151004833118708998612454041255393121328792594


In [6]:
# 4) Sign m using your private key and a randomly chosen nonce k. produce (r, s, h, PubKey)
k = random_uint256()
R = k * G
r = R.x
s = (h + r*p % order) * pow(k, -1, order) % order

print("p:",p)
print("h:",h)
print("r:",r)
print("s:",s)

p: 72329000536779791609889445150063805330999049088740615955324526097706168154835
h: 38294091763788184178067574739767151004833118708998612454041255393121328792594
r: 40336420830748971251530934757786039359921062277952434889668245339754048864222
s: 20392598222657222564222667921709259191778336399690708914631643118576140309770


In [7]:
# 5) Verify (r, s, h, PubKey) is valid
a, b = cv.a, cv.b
t = (pow(r, 3, field) + b) % field
R_y = list(sqrtmod_prime_power(t, field, 1))[0]

R = Point(r, R_y, cv)

P_recovered = (s * pow(r,-1,order)% order) * R  - (pow(r,-1,order) * h % order) * G
print("P recovered:", P_recovered)
print("Addr recovered:", calc_eth_addr(P_recovered))

P recovered: (0xa0fe012cafe7b9eb78c6589d9abc1d6feebf53b8f63b8f5dc65f31b66860fbf1 , 0xc47296ce065b9f7367dc729193cec3bf66f22fb2249b21c2ba19a4b06009d5b2)
Addr recovered: 0x9ebd5ccf93049c6acc236dca6664796bd3acaa5e


In [8]:
match = calc_eth_addr(P) == calc_eth_addr(P_recovered)
print("Does it work?", match)

Does it work? True
